# 5. Semantic image segmentation with DeepLabV3
Segmentation assigns a class to each pixel. We'll run a pretrained Torchvision DeepLabV3 model on a sample image.

In [ ]:
!pip -q install torch torchvision matplotlib pillow requests

In [ ]:
import torch, torchvision, numpy as np, matplotlib.pyplot as plt
from torchvision.models.segmentation import deeplabv3_resnet50, DeepLabV3_ResNet50_Weights
from PIL import Image
from io import BytesIO
import requests
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
weights=DeepLabV3_ResNet50_Weights.DEFAULT
model=deeplabv3_resnet50(weights=weights).to(device).eval()
preprocess=weights.transforms()
url="https://images.cocodataset.org/val2017/000000039769.jpg"
image=Image.open(BytesIO(requests.get(url,timeout=30).content)).convert("RGB")
tensor=preprocess(image).unsqueeze(0).to(device)
with torch.no_grad(): logits=model(tensor)["out"][0]
mask=logits.argmax(0).cpu().numpy()
cats=weights.meta["categories"]
unique, counts=np.unique(mask,return_counts=True)
print("Predicted classes:", [(cats[int(i)],int(c)) for i,c in zip(unique,counts) if int(i)<len(cats)])

In [ ]:
fig,ax=plt.subplots(1,2,figsize=(14,6))
ax[0].imshow(image); ax[0].set_title("Original image"); ax[0].axis("off")
ax[1].imshow(mask,cmap="tab20"); ax[1].set_title("Predicted pixel classes"); ax[1].axis("off")
plt.show()

**Expected output:** a colored pixel-level mask. This model uses the PASCAL VOC label set; results are not medical-grade. For custom segmentation, use annotated masks and fine-tune U-Net or DeepLabV3.